# Vetting Statistics

This notebook summarizes retrieved vetting fields from MALCA vetted parquet products and plots their distributions.

Numeric fields are shown as histograms, boolean fields as bar charts, categorical fields as top-N bar charts, and free-text identifier columns are treated as completeness metrics rather than forced into fake histograms.


In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'legend.frameon': True,
})


## Configuration

- Leave `VETTED_FILES` empty to auto-discover `lc_events_vetted*.parquet` under `../../output/runs`.
- Standard pipeline outputs only include columns produced by `vet_candidates(...)`; optional follow-up light-curve columns may be absent.


In [ ]:
RUNS_ROOT = Path('../../output/runs')
VETTED_FILES = []
TOP_N_CATEGORIES = 12
PERCENTILE_CLIP = (0.01, 0.99)

BASE_COLORS = [
    '#1b9e77', '#d95f02', '#7570b3', '#e7298a',
    '#66a61e', '#e6ab02', '#a6761d', '#666666',
]

KNOWN_VETTING_COLUMNS = [
    'vetting_likely_known',
    'simbad_main_id', 'simbad_otype', 'simbad_nbref', 'simbad_sep_arcsec',
    'gaia_var_flag', 'gaia_var_class', 'gaia_var_score',
    'gaia_eb_period', 'gaia_eb_morph', 'gaia_eb_global_ranking',
    'gaia_epoch_available', 'gaia_epoch_n_obs', 'gaia_epoch_g_range',
    'asassn_var_name', 'asassn_var_type', 'asassn_var_period',
    'ztf_var_type', 'ztf_var_period', 'ztf_var_amp',
    'tns_name', 'tns_type', 'tns_redshift', 'tns_disc_date',
    'alerce_oid', 'alerce_ndet', 'alerce_lc_class', 'alerce_lc_prob',
    'alerce_stamp_class', 'alerce_stamp_prob',
    'xray_det', 'xray_flux', 'xray_sep_arcsec',
    'vsx_class', 'vsx_sep_arcsec',
    'sfr_name', 'sfr_sep_arcmin',
    'cluster_name', 'cluster_dist_pc',
    'banyan_best_assoc', 'banyan_field_prob',
    'yso_class',
    'iphas_ha_excess',
    'pm_cluster_offset_sigma',
    'atlas_has_phot', 'atlas_n_det_cyan', 'atlas_n_det_orange',
    'atlas_cyan_range', 'atlas_orange_range',
    'neowise_n_epochs', 'neowise_w1_range', 'neowise_w2_range',
]

VETTING_PREFIXES = (
    'vetting_', 'simbad_', 'gaia_', 'asassn_', 'ztf_', 'tns_',
    'alerce_', 'xray_', 'vsx_', 'sfr_', 'cluster_', 'banyan_',
    'yso_', 'iphas_', 'pm_', 'atlas_', 'neowise_',
)

BOOLEAN_COLUMNS = {
    'vetting_likely_known',
    'gaia_var_flag',
    'gaia_epoch_available',
    'xray_det',
    'atlas_has_phot',
    'iphas_ha_excess',
}

CATEGORICAL_COLUMNS = {
    'simbad_otype',
    'gaia_var_class',
    'gaia_eb_morph',
    'asassn_var_type',
    'ztf_var_type',
    'tns_type',
    'alerce_lc_class',
    'alerce_stamp_class',
    'vsx_class',
    'sfr_name',
    'cluster_name',
    'banyan_best_assoc',
    'yso_class',
}

TEXT_COLUMNS = {
    'simbad_main_id',
    'asassn_var_name',
    'tns_name',
    'tns_disc_date',
    'alerce_oid',
}

MODULE_MARKERS = {
    'Likely known': 'vetting_likely_known',
    'SIMBAD': 'simbad_main_id',
    'Gaia variability': 'gaia_var_flag',
    'Gaia EB': 'gaia_eb_period',
    'Gaia epoch photometry': 'gaia_epoch_available',
    'ASAS-SN variables': 'asassn_var_type',
    'ZTF variables': 'ztf_var_type',
    'TNS': 'tns_name',
    'ALeRCE': 'alerce_oid',
    'eROSITA': 'xray_det',
    'ATLAS': 'atlas_has_phot',
    'PM consistency': 'pm_cluster_offset_sigma',
    'NEOWISE': 'neowise_n_epochs',
}

LOG10_COLUMNS = {
    'simbad_nbref',
    'simbad_sep_arcsec',
    'gaia_eb_period',
    'gaia_epoch_n_obs',
    'gaia_epoch_g_range',
    'asassn_var_period',
    'ztf_var_period',
    'ztf_var_amp',
    'alerce_ndet',
    'xray_flux',
    'xray_sep_arcsec',
    'atlas_n_det_cyan',
    'atlas_n_det_orange',
    'atlas_cyan_range',
    'atlas_orange_range',
    'neowise_n_epochs',
    'neowise_w1_range',
    'neowise_w2_range',
}


In [ ]:
def infer_mag_bin_label(run_dir: Path) -> str:
    run_params = run_dir / 'run_params.json'
    if run_params.exists():
        try:
            params = json.loads(run_params.read_text())
            mag_bin = params.get('mag_bin')
            if mag_bin:
                return str(mag_bin)
            mag_min = params.get('mag_min')
            mag_max = params.get('mag_max')
            if mag_min is not None and mag_max is not None:
                return f'{float(mag_min):g}-{float(mag_max):g}'
        except Exception:
            pass

    match = re.search(r'(\d+(?:\.\d+)?)_(\d+(?:\.\d+)?)', run_dir.name)
    if match:
        return f'{float(match.group(1)):g}-{float(match.group(2)):g}'
    return run_dir.name


def sort_labels(labels):
    def key(label):
        match = re.match(r'^(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?)$', str(label))
        if match:
            return (0, float(match.group(1)), float(match.group(2)), str(label))
        return (1, str(label))

    return sorted(labels, key=key)


def discover_vetted_files() -> pd.DataFrame:
    rows = []
    if VETTED_FILES:
        paths = [Path(p) for p in VETTED_FILES]
    else:
        paths = sorted(RUNS_ROOT.glob('*/results/lc_events_vetted*.parquet'))

    for path in paths:
        if not path.exists():
            continue
        run_dir = path.parent.parent
        rows.append({
            'run_name': run_dir.name,
            'mag_bin': infer_mag_bin_label(run_dir),
            'candidate_file': path,
        })

    if not rows:
        raise FileNotFoundError(f'No vetted parquet files found under {RUNS_ROOT}')
    return pd.DataFrame(rows)


def load_vetting_frame(file_manifest: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for row in file_manifest.itertuples(index=False):
        df = pd.read_parquet(row.candidate_file)
        df = df.copy()
        df['run_name'] = row.run_name
        df['mag_bin'] = row.mag_bin
        df['candidate_file'] = str(row.candidate_file)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def present_count(series: pd.Series) -> int:
    if pd.api.types.is_bool_dtype(series):
        return int(series.fillna(False).sum())
    if pd.api.types.is_numeric_dtype(series):
        s = pd.to_numeric(series, errors='coerce')
        if set(pd.unique(s.dropna())) <= {0.0, 1.0}:
            return int(s.fillna(0).sum())
        return int(s.notna().sum())
    return int(series.fillna('').astype(str).str.strip().ne('').sum())


def numeric_series(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce')
    return s[np.isfinite(s)]


def classify_column(col: str, series: pd.Series) -> str:
    if col in BOOLEAN_COLUMNS:
        return 'boolean'
    if col in CATEGORICAL_COLUMNS:
        return 'categorical'
    if col in TEXT_COLUMNS:
        return 'text'
    if pd.api.types.is_numeric_dtype(series):
        return 'numeric'
    clean = series.fillna('').astype(str).str.strip()
    clean = clean[clean != '']
    if clean.empty:
        return 'text'
    if clean.nunique() <= TOP_N_CATEGORIES:
        return 'categorical'
    return 'text'


def numeric_group(col: str) -> str:
    if col.startswith('simbad_') or col.endswith('_sep_arcsec') or col.endswith('_sep_arcmin'):
        return 'Match quality and separation'
    if col.startswith('gaia_'):
        return 'Gaia retrieved stats'
    if col.startswith('asassn_') or col.startswith('ztf_'):
        return 'ASAS-SN and ZTF'
    if col.startswith('tns_') or col.startswith('alerce_'):
        return 'TNS and ALeRCE'
    if col.startswith('xray_') or col.startswith('pm_') or col.startswith('banyan_'):
        return 'X-ray and environment'
    if col.startswith('atlas_') or col.startswith('neowise_'):
        return 'ATLAS and NEOWISE'
    return 'Other numeric vetting'


NUMERIC_GROUP_ORDER = [
    'Match quality and separation',
    'Gaia retrieved stats',
    'ASAS-SN and ZTF',
    'TNS and ALeRCE',
    'X-ray and environment',
    'ATLAS and NEOWISE',
    'Other numeric vetting',
]


def transform_numeric(col: str, series: pd.Series) -> tuple[pd.Series, str, str]:
    s = numeric_series(series)
    xlabel = col
    transform = 'linear'
    if col in LOG10_COLUMNS:
        s = s[s > 0]
        if not s.empty:
            s = np.log10(s)
            xlabel = f'log10({col})'
            transform = 'log10'
    return s, xlabel, transform


def clip_series(series: pd.Series) -> pd.Series:
    if series.empty:
        return series
    lo = series.quantile(PERCENTILE_CLIP[0])
    hi = series.quantile(PERCENTILE_CLIP[1])
    clipped = series[(series >= lo) & (series <= hi)]
    return clipped if not clipped.empty else series


def palette_for(labels: list[str]) -> dict[str, str]:
    ordered = sort_labels(labels)
    return {label: BASE_COLORS[i % len(BASE_COLORS)] for i, label in enumerate(ordered)}


def build_inventory(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        col_type = classify_column(col, df[col])
        rows.append({
            'column': col,
            'column_type': col_type,
            'group': numeric_group(col) if col_type == 'numeric' else '',
            'dtype': str(df[col].dtype),
            'present_rows': present_count(df[col]),
            'present_fraction': present_count(df[col]) / len(df) if len(df) else np.nan,
            'n_unique_non_null': int(df[col].dropna().astype(str).nunique()),
        })
    inventory = pd.DataFrame(
        rows,
        columns=['column', 'column_type', 'group', 'dtype', 'present_rows', 'present_fraction', 'n_unique_non_null'],
    )
    if inventory.empty:
        return inventory
    return inventory.sort_values(['column_type', 'group', 'column']).reset_index(drop=True)


def plot_numeric_group(df: pd.DataFrame, cols: list[str], title: str, split_col: str = 'mag_bin') -> None:
    if not cols:
        return

    labels = sort_labels(df[split_col].dropna().unique())
    palette = palette_for(labels)
    ncols = min(3, len(cols))
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.8 * nrows), squeeze=False)
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.02)

    for idx, col in enumerate(cols):
        ax = axes[idx // ncols, idx % ncols]
        for label in labels:
            series, xlabel, _ = transform_numeric(col, df.loc[df[split_col] == label, col])
            series = clip_series(series)
            if series.empty:
                continue
            ax.hist(
                series,
                bins=40,
                density=True,
                histtype='step',
                linewidth=1.8,
                color=palette[label],
                label=label,
            )
        ax.set_title(col, fontsize=10)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel('density', fontsize=9)
        ax.tick_params(labelsize=8)
        if idx == 0:
            ax.legend(fontsize=8, title=split_col)

    for idx in range(len(cols), nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.tight_layout()
    plt.show()


def plot_boolean_overview(df: pd.DataFrame, cols: list[str]) -> None:
    if not cols:
        return
    rows = []
    for col in cols:
        s = pd.to_numeric(df[col], errors='coerce').fillna(0)
        rows.append({'column': col, 'true_fraction': float(s.mean()), 'true_count': int(s.sum())})
    plot_df = pd.DataFrame(rows).sort_values('true_fraction', ascending=False)
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.45 * len(plot_df))))
    ax.barh(plot_df['column'], plot_df['true_fraction'], color='#4c78a8')
    ax.set_xlabel('fraction of rows flagged true')
    ax.set_ylabel('')
    ax.set_title('Boolean vetting fields')
    ax.invert_yaxis()
    plt.show()


def plot_categorical_grid(df: pd.DataFrame, cols: list[str], top_n: int = TOP_N_CATEGORIES) -> None:
    if not cols:
        return
    ncols = min(2, len(cols))
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 3.8 * nrows), squeeze=False)
    fig.suptitle('Categorical vetting fields', fontsize=15, fontweight='bold', y=1.02)

    for idx, col in enumerate(cols):
        ax = axes[idx // ncols, idx % ncols]
        counts = (
            df[col]
            .fillna('')
            .astype(str)
            .str.strip()
            .replace('', np.nan)
            .dropna()
            .value_counts()
            .head(top_n)
            .sort_values(ascending=True)
        )
        if counts.empty:
            ax.set_visible(False)
            continue
        ax.barh(counts.index, counts.values, color='#59a14f')
        ax.set_title(col, fontsize=10)
        ax.set_xlabel('count')
        ax.tick_params(labelsize=8)

    for idx in range(len(cols), nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.tight_layout()
    plt.show()


In [ ]:
file_manifest = discover_vetted_files()
display(file_manifest)

df_vet = load_vetting_frame(file_manifest)
print(f'Loaded {len(df_vet):,} vetted rows from {file_manifest.shape[0]} file(s).')


In [ ]:
discovered_vetting_cols = sorted([
    c for c in df_vet.columns
    if c in KNOWN_VETTING_COLUMNS or c.startswith(VETTING_PREFIXES)
])

inventory = build_inventory(df_vet, discovered_vetting_cols)
INVENTORY_MAP = inventory.set_index('column').to_dict(orient='index')

print(f'{len(discovered_vetting_cols)} vetting columns available in the loaded data')
display(inventory)


## Module Hit Rates

This mirrors the quick vetting summary and makes it obvious which modules actually returned matches in the current vetted products.


In [ ]:
module_rows = []
for module_name, marker in MODULE_MARKERS.items():
    if marker not in df_vet.columns:
        continue
    hits = present_count(df_vet[marker])
    module_rows.append({
        'module': module_name,
        'marker_column': marker,
        'hits': hits,
        'fraction': hits / len(df_vet) if len(df_vet) else np.nan,
    })

module_summary = pd.DataFrame(
    module_rows,
    columns=['module', 'marker_column', 'hits', 'fraction'],
)
if not module_summary.empty:
    module_summary = module_summary.sort_values('fraction', ascending=False)
display(module_summary)


## Completeness By Column Type

The audit below separates numeric histogram targets from boolean, categorical, and text-only vetting fields.


In [ ]:
completeness = (
    inventory.groupby('column_type')['column']
    .count()
    .rename('n_columns')
    .reset_index()
    .sort_values('column_type')
)
display(completeness)


## Boolean Fields

Boolean vetting fields are better represented as hit-rate bar charts than histograms.


In [ ]:
boolean_cols = [c for c in discovered_vetting_cols if INVENTORY_MAP[c]['column_type'] == 'boolean']
plot_boolean_overview(df_vet, boolean_cols)


## Categorical Fields

Top-N categories are shown for categorical vetting outputs such as SIMBAD types or Gaia variable classes.


In [ ]:
categorical_cols = [c for c in discovered_vetting_cols if INVENTORY_MAP[c]['column_type'] == 'categorical']
plot_categorical_grid(df_vet, categorical_cols)


## Numeric Retrieved Stats

Every numeric vetting field available in the loaded data is plotted below. Long-tailed positive quantities use `log10(...)` automatically.


In [ ]:
numeric_cols = [c for c in discovered_vetting_cols if INVENTORY_MAP[c]['column_type'] == 'numeric']
for group_name in NUMERIC_GROUP_ORDER:
    group_cols = [c for c in numeric_cols if INVENTORY_MAP[c]['group'] == group_name]
    if group_cols:
        plot_numeric_group(df_vet, group_cols, title=group_name)


## Text-Only Vetting Fields

Free-text identifier fields are still useful as completeness signals even though they are not histogram targets.


In [ ]:
text_cols = [c for c in discovered_vetting_cols if INVENTORY_MAP[c]['column_type'] == 'text']
text_completeness = pd.DataFrame(
    [
        {
            'column': col,
            'present_rows': present_count(df_vet[col]),
            'present_fraction': present_count(df_vet[col]) / len(df_vet) if len(df_vet) else np.nan,
        }
        for col in text_cols
    ],
    columns=['column', 'present_rows', 'present_fraction'],
)
if not text_completeness.empty:
    text_completeness = text_completeness.sort_values('present_fraction', ascending=False)
display(text_completeness)


## Notes

- Missing vetting columns usually mean the corresponding module was disabled or did not yield matches.
- `atlas_*` fields are often sparse unless ATLAS querying was enabled with a token.
- This notebook only shows fields present in the loaded vetted parquet files, so optional external follow-up columns may not appear.
